In [15]:
import kaggle_benchmarks as kbench        
from dataclasses import dataclass

In [31]:
@dataclass
class HangmanResponse:
    guessed_character: str

@kbench.task(name="hangman_solver", description="Solve a Hangman Puzzle")
def solve_hangman_word(llm, word:str):
    livesLeft = 6

    EMPTY = "_"
    current_guess = [EMPTY] * len(word)
    incorrect_chars = []

    turn_number = 1;
    initial_prompt = f"""
    You are a Hangman Game Expert. Your goal is to win the game using optimal linguistic strategy.
	RULES:
	1. You are provided with the current state of the word (e.g., "h _ _ p _") and a list of incorrect guesses.
	2. You must analyze the word length and patterns to choose the most likely letter.

    Current Guess: {" ".join(current_guess)}
    Lives Left: {livesLeft}
    Incorrect Guesses: {",".join(incorrect_chars)}
    """

    while livesLeft > 0:        
        if turn_number == 0:
            # kinda like system prompting
            prompt = initial_prompt
        else:
            prompt = f"""
            Current Guess: {" ".join(current_guess)}
            Lives Left: {livesLeft}
            Incorrect Chars: {",".join(incorrect_chars)}
            """
        response = llm.prompt(prompt, schema=HangmanResponse)
        
        if not response.guessed_character in word:
            livesLeft -= 1
            incorrect_chars.append(response.guessed_character)
        else:
            # construct a new guess str
            for idx, char in enumerate(word):
                if current_guess[idx] == EMPTY and response.guessed_character == char:
                    current_guess[idx] = response.guessed_character
                    continue
            guessed_word = "".join(current_guess)
            if guessed_word == word:
                break
        turn_number += 1
   
    # handle assertions

    final_word = "".join(current_guess)
    kbench.assertions.assert_equal(
        final_word,
        word,
        expectation=f"LLM must guess the correct word i.e {word}, But instead guessed {final_word}",
    )

BokehModel(combine_events=True, render_bundle={'docs_json': {'31b051bd-ae69-46db-a937-eec96d1b4131': {'version…

In [30]:
solve_hangman_word.run(llm=kbench.llm,word="cow")

BokehModel(combine_events=True, render_bundle={'docs_json': {'184b7749-f227-4217-a471-84e36feeb57c': {'version…

BokehModel(combine_events=True, render_bundle={'docs_json': {'a58ad48d-2cbb-4d50-ba19-29188ab3f946': {'version…